# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
import getpass
import duckdb

# Get the Hugging Face token securely
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

HF_TOKEN = get_hf_token()

# Create DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 development partition
FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection established.")
print("Using March 2026 development data.")

DuckDB connection established.
Using March 2026 development data.


### Feature vector

For the March 2026 development window, I will represent each client-content pair using five performance features: impressions, clicks, CTR, average position, and position volatility.

The features are aggregated from the same March 2026 window so that each observation has one consistent feature vector. Client and content IDs are retained only for identification and grouping, not as clustering features.

Rows without available GSC data are not treated as zero performance; GSC-derived metrics are calculated only where the relevant data is available.

In [4]:
# Inspect the actual columns available in the March 2026 fact table

schema = con.sql(f"""
    DESCRIBE SELECT *
    FROM {FACT_DAILY}
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [5]:
# Show all column names in the March 2026 fact table

for column in schema["column_name"]:
    print(column)


report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


In [6]:
# Build the first five clustering features from March 2026

feature_vector = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(
            NULLIF(gsc_avg_position, 0)
        ) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

display(feature_vector.head())

print("Number of client-content pairs:", len(feature_vector))
print("Number of clustering features:", 5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_volatility
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.888929,2.119233
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.240351,1.376604
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,7.061594,3.686090
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.141844,6.155424,3.892530
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.343567,14.439705


Number of client-content pairs: 176738
Number of clustering features: 5


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.